# 🔍 PNCP — Extração e Análise de Itens: Switch

Este notebook:
1. Lê uma planilha CSV/Excel com a coluna `ID PNCP`
2. Para cada ID, consulta a API do PNCP e busca itens de switch
3. Aplica **dois algoritmos** para classificar se o switch é o item principal:
   - **Algoritmo 1:** Regex (regras textuais)
   - **Algoritmo 2:** Ollama (LLM local)
4. Extrai marca/modelo via regex e Ollama
5. Gera relatório comparativo entre os dois algoritmos
6. Exporta CSV com os resultados

## ⚙️ Célula 1 — Configurações

In [ ]:
# ============================================================
#  CONFIGURAÇÕES — edite aqui antes de rodar
# ============================================================

# Caminho para sua planilha (CSV ou Excel)
ARQUIVO_ENTRADA = "licitacoes.xlsx"   # ou "licitacoes.csv"

# Nome da coluna com o ID PNCP
COLUNA_ID_PNCP = "ID PNCP"

# Arquivos de saída
ARQUIVO_SAIDA_DADOS    = "switch_dados_extraidos.xlsx"
ARQUIVO_SAIDA_RELATORIO = "switch_relatorio_comparativo.xlsx"

# Ollama
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "phi3:mini"   # troque pelo modelo que você tem instalado (ex: mistral, phi3)

# API PNCP
## Rotas corretas (seguindo o ata_downloader): a base já inclui /v1
PNCP_BASE_URL = "https://pncp.gov.br/api/pncp/v1"

# Pausa entre chamadas à API (segundos) — evita rate limit
PAUSA_SEGUNDOS = 0.5

print("✅ Configurações carregadas")

✅ Configurações carregadas


## 📦 Célula 2 — Instalação e Imports

In [15]:
import subprocess, sys, importlib
from typing import Optional

def instalar(pacote: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

# Dependências básicas do notebook
for pkg in ["requests", "pandas", "openpyxl", "tqdm"]:
    instalar(pkg)

# Widgets melhoram a barra de progresso em ambiente notebook; se não der, seguimos sem eles.
try:
    instalar("ipywidgets")
except Exception:
    pass

import re
import time
import json
import requests
import pandas as pd

# tqdm.auto escolhe automaticamente a melhor implementação (notebook ou terminal)
from tqdm.auto import tqdm

from IPython.display import display

print("✅ Imports OK")

✅ Imports OK


## 🔧 Célula 3 — Funções: Parse do ID PNCP e API

In [ ]:
import random
from urllib.parse import urlparse

# Cache simples em memória para evitar repetir GETs na API durante reexecuções
_API_CACHE: dict[str, object] = {}

def _is_cacheable(url: str) -> bool:
    try:
        p = urlparse(url)
        return p.scheme in {"http", "https"} and bool(p.netloc)
    except Exception:
        return False

def api_get(url: str, *, timeout: int = 20, retries: int = 3, backoff_base: float = 0.6,
            use_cache: bool = True) -> dict | list | None:
    """
    Faz GET na API do PNCP com tratamento de erros, retry/backoff e cache opcional.

    Retorna JSON (dict/list) quando houver sucesso (HTTP 200) e o corpo for JSON válido; caso contrário None.
    Quando falhar, você pode capturar detalhes usando api_get_detalhado(...).
    """
    if use_cache and _is_cacheable(url) and url in _API_CACHE:
        return _API_CACHE[url]  # type: ignore[return-value]

    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers={"accept": "application/json"}, timeout=timeout)
            if r.status_code == 200:
                try:
                    data = r.json()
                except Exception:
                    data = None
                if use_cache and data is not None and _is_cacheable(url):
                    _API_CACHE[url] = data
                return data
            # 429/5xx: tenta novamente; 4xx (exceto 429) costuma ser terminal
            if r.status_code in (429, 500, 502, 503, 504) and attempt < retries:
                sleep_s = backoff_base * (2 ** attempt) + random.uniform(0, 0.2)
                time.sleep(sleep_s)
                continue
            return None
        except (requests.Timeout, requests.ConnectionError) as e:
            if attempt < retries:
                sleep_s = backoff_base * (2 ** attempt) + random.uniform(0, 0.2)
                time.sleep(sleep_s)
                continue
            return None
        except Exception:
            return None

def api_get_detalhado(url: str, *, timeout: int = 20, retries: int = 3, backoff_base: float = 0.6,
                     use_cache: bool = True) -> tuple[dict | list | None, dict]:
    """
    Versão detalhada para diagnóstico: retorna (data, meta).
    meta inclui: ok, status_code, erro, url, content_type, texto_curto
    """
    if use_cache and _is_cacheable(url) and url in _API_CACHE:
        return _API_CACHE[url], {"ok": True, "status_code": 200, "erro": None, "url": url, "cached": True}  # type: ignore[misc]

    last_meta: dict = {"ok": False, "status_code": None, "erro": None, "url": url, "cached": False}
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers={"accept": "application/json"}, timeout=timeout)
            ct = r.headers.get("content-type", "")
            texto_curto = (r.text or "")[:250] if r is not None else ""
            if r.status_code == 200:
                try:
                    data = r.json()
                except Exception as e:
                    return None, {"ok": False, "status_code": 200, "erro": f"JSON inválido: {e}", "url": url, "content_type": ct, "texto_curto": texto_curto, "cached": False}
                if use_cache and _is_cacheable(url):
                    _API_CACHE[url] = data
                return data, {"ok": True, "status_code": 200, "erro": None, "url": url, "content_type": ct, "texto_curto": None, "cached": False}
            last_meta = {"ok": False, "status_code": r.status_code, "erro": None, "url": url, "content_type": ct, "texto_curto": texto_curto, "cached": False}
            if r.status_code in (429, 500, 502, 503, 504) and attempt < retries:
                sleep_s = backoff_base * (2 ** attempt) + random.uniform(0, 0.2)
                time.sleep(sleep_s)
                continue
            return None, last_meta
        except (requests.Timeout, requests.ConnectionError) as e:
            last_meta = {"ok": False, "status_code": None, "erro": str(e), "url": url, "cached": False}
            if attempt < retries:
                sleep_s = backoff_base * (2 ** attempt) + random.uniform(0, 0.2)
                time.sleep(sleep_s)
                continue
            return None, last_meta
        except Exception as e:
            return None, {"ok": False, "status_code": None, "erro": str(e), "url": url, "cached": False}

def parse_id_pncp(id_pncp: str) -> dict | None:
    """
    Converte '14226731000164-1-000018/2025' em
    {'cnpj': '14226731000164', 'sequencial': 18, 'ano': 2025}

    Observação: segue o mesmo formato do ata_downloader, aceitando sequencial 1..6 dígitos.
    """
    try:
        id_pncp = str(id_pncp).strip()
        m = re.match(r"^(\d{14})-1-(\d{1,6})/(\d{4})$", id_pncp)
        if not m:
            return None
        cnpj = m.group(1)
        sequencial = int(m.group(2))
        ano = int(m.group(3))
        return {'cnpj': cnpj, 'sequencial': sequencial, 'ano': ano}
    except Exception:
        return None

def buscar_contratacao(cnpj, ano, sequencial) -> dict | None:
    # rota correta: {PNCP_BASE_URL}/orgaos/{cnpj}/compras/{ano}/{sequencial}
    url = f"{PNCP_BASE_URL}/orgaos/{cnpj}/compras/{ano}/{sequencial}"
    data =  api_get(url)
    return data if isinstance(data, dict) else None

def buscar_itens(cnpj, ano, sequencial) -> list:
    # rota correta: {PNCP_BASE_URL}/orgaos/{cnpj}/compras/{ano}/{sequencial}/itens
    url = f"{PNCP_BASE_URL}/orgaos/{cnpj}/compras/{ano}/{sequencial}/itens"
    resultado = api_get(url)
    if isinstance(resultado, list):
        return resultado
    # Alguns retornos vêm paginados
    if isinstance(resultado, dict) and 'data' in resultado:
        return resultado['data']
    return []

def buscar_resultado_item(cnpj, ano, sequencial, numero_item) -> list:
    # rota correta: {PNCP_BASE_URL}/orgaos/{cnpj}/compras/{ano}/{sequencial}/itens/{numero_item}/resultados
    url = f"{PNCP_BASE_URL}/orgaos/{cnpj}/compras/{ano}/{sequencial}/itens/{numero_item}/resultados"
    resultado = api_get(url)
    if isinstance(resultado, list):
        return resultado
    return []

print("✅ Funções de API carregadas (rotas PNCP corretas + retry/cache/diagnóstico)")

✅ Funções de API carregadas (com retry/cache/diagnóstico)


## 🔤 Célula 4 — Algoritmo 1: Regex

In [17]:
# Padrões que indicam que switch é COMPONENTE (não o item principal)
PADROES_COMPONENTE = [
    r'\bcom\s+switch\b',           # "roteador com switch"
    r'\bintegra\w*\s+switch\b',    # "switch integrado"
    r'\binclu\w*\s+switch\b',      # "inclui switch"
    r'\bkit\b.*\bswitch\b',        # "kit ... switch"
    r'\bswitch\b.*\bintegra\w*\b', # "switch integrado"
    r'\bporta\s+switch\b',         # "porta switch"
    r'rack.*switch',               # "rack com switch"
    r'switch.*embutid\w*',         # "switch embutido"
    r'patch\s+panel.*switch',      # "patch panel e switch"
]

# Padrões que confirmam switch como item principal
PADROES_PRINCIPAL = [
    r'^switch\b',                           # começa com switch
    r'^switch\s+\d+\s+portas\b',           # "switch 8 portas"
    r'^switch\s+(gerenci\w+|l[23]|poe\b)',  # switch gerenciável, L2, L3, PoE
    r'\bswitch\b.*\bportas\b',             # switch ... portas
    r'\bswitch\b.*\bmbps\b',               # switch ... mbps
    r'\bswitch\b.*\bgbps\b',               # switch ... gbps
    r'\bswitch\b.*\b10/100\b',             # switch 10/100
    r'\bswitch\b.*\b1000\b',               # switch 1000
]


def regex_eh_switch_principal(descricao: str) -> bool:
    """
    Retorna True se a descrição indica que switch é o item principal.
    Lógica: deve conter 'switch', não ter padrão de componente, e
    idealmente ter padrão de principal.
    """
    texto = descricao.lower().strip()

    if 'switch' not in texto:
        return False

    # Se tiver padrão de componente → não é principal
    for padrao in PADROES_COMPONENTE:
        if re.search(padrao, texto, re.IGNORECASE):
            return False

    # Se tiver padrão de principal → é principal
    for padrao in PADROES_PRINCIPAL:
        if re.search(padrao, texto, re.IGNORECASE):
            return True

    # Caso ambíguo: 'switch' existe mas sem padrão claro
    # Heurística: se switch é uma das primeiras 2 palavras → principal
    palavras = texto.split()
    if palavras and palavras[0] == 'switch':
        return True
    if len(palavras) > 1 and palavras[1] == 'switch':
        return True

    return False


def regex_extrair_marca_modelo(descricao: str) -> dict:
    """Extrai marca e modelo via regex a partir da descrição."""
    resultado = {'marca_regex': None, 'modelo_regex': None}
    texto = descricao.strip()

    # Padrão explícito: MARCA: X / MODELO: Y
    m = re.search(r'marca[:\s]+([A-Z][A-Z0-9\s\-\.]+?)(?:\s+modelo|\s*[;\/]|$)',
                  texto, re.IGNORECASE)
    if m:
        resultado['marca_regex'] = m.group(1).strip()

    m = re.search(r'modelo[:\s]+([A-Z0-9][A-Z0-9\s\-\/\.]+?)(?:\s*[;\/]|\s+[A-Z]{4,}|$)',
                  texto, re.IGNORECASE)
    if m:
        resultado['modelo_regex'] = m.group(1).strip()

    # Marcas conhecidas de switch — fallback quando não tem campo explícito
    MARCAS_CONHECIDAS = [
        'INTELBRAS', 'CISCO', 'HP', 'DELL', 'NETGEAR', 'TP-LINK', 'TPLINK',
        'D-LINK', 'DLINK', 'UBIQUITI', 'MIKROTIK', 'JUNIPER', 'HUAWEI',
        'EXTREME', '3COM', 'ARUBA', 'ZYXEL', 'LINKSYS'
    ]
    if not resultado['marca_regex']:
        for marca in MARCAS_CONHECIDAS:
            if marca.lower() in texto.lower():
                resultado['marca_regex'] = marca
                break

    return resultado


# --- Teste rápido ---
testes = [
    ("SWITCH 08 PORTAS 10/100/1000 MBPS", True),
    ("SWITCH 16 PORTAS 10/100/1000 MBPS", True),
    ("ROTEADOR 1200 MBPS COM SWITCH INTEGRADO 4 PORTAS", False),
    ("KIT RACK COM PATCH PANEL E SWITCH", False),
    ("SWITCH GERENCIÁVEL L2 24 PORTAS INTELBRAS SG 2404 QR", True),
    ("MOUSE USB", False),
]
print("Testes Algoritmo 1 (Regex):")
print(f"{'Descrição':<55} {'Esperado':>8} {'Obtido':>8} {'OK':>4}")
print("-" * 80)
for desc, esperado in testes:
    obtido = regex_eh_switch_principal(desc)
    ok = "✅" if obtido == esperado else "❌"
    print(f"{desc[:54]:<55} {str(esperado):>8} {str(obtido):>8} {ok:>4}")

Testes Algoritmo 1 (Regex):
Descrição                                               Esperado   Obtido   OK
--------------------------------------------------------------------------------
SWITCH 08 PORTAS 10/100/1000 MBPS                           True     True    ✅
SWITCH 16 PORTAS 10/100/1000 MBPS                           True     True    ✅
ROTEADOR 1200 MBPS COM SWITCH INTEGRADO 4 PORTAS           False    False    ✅
KIT RACK COM PATCH PANEL E SWITCH                          False    False    ✅
SWITCH GERENCIÁVEL L2 24 PORTAS INTELBRAS SG 2404 QR        True     True    ✅
MOUSE USB                                                  False    False    ✅


## 🤖 Célula 5 — Algoritmo 2: Ollama (LLM Local)

In [18]:
def ollama_classificar_e_extrair(descricao: str) -> dict:
    """
    Usa Ollama para:
    1. Classificar se switch é item principal
    2. Extrair marca e modelo
    Retorna dict com: eh_principal_ollama, marca_ollama, modelo_ollama
    """
    prompt = f"""Você é um especialista em licitações públicas brasileiras.
Analise a descrição de um item de licitação abaixo.

Responda APENAS com um JSON válido, sem texto adicional, sem markdown, sem explicações.
Formato exato:
{{"principal": true/false, "marca": "string ou null", "modelo": "string ou null"}}

Regras:
- "principal": true SOMENTE se o switch for o produto sendo licitado (ex: "SWITCH 8 PORTAS").
  false se switch for componente de outro produto (ex: "ROTEADOR COM SWITCH INTEGRADO") ou parte de kit.
- "marca": nome da fabricante se mencionado, senão null.
- "modelo": código/número do modelo se mencionado, senão null.

Descrição: {descricao}"""

    try:
        r = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
            timeout=30
        )
        if r.status_code != 200:
            return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                    'erro_ollama': f'HTTP {r.status_code}'}

        resposta_texto = r.json().get('response', '').strip()

        # Limpa possíveis blocos markdown que o modelo possa gerar
        resposta_texto = re.sub(r'```json|```', '', resposta_texto).strip()

        # Extrai o JSON da resposta
        match = re.search(r'\{.*\}', resposta_texto, re.DOTALL)
        if not match:
            return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                    'erro_ollama': f'JSON não encontrado: {resposta_texto[:100]}'}

        dados = json.loads(match.group())
        return {
            'eh_principal_ollama': bool(dados.get('principal')),
            'marca_ollama':        dados.get('marca'),
            'modelo_ollama':       dados.get('modelo'),
            'erro_ollama':         None
        }

    except json.JSONDecodeError as e:
        return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                'erro_ollama': f'JSON inválido: {str(e)}'}
    except Exception as e:
        return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                'erro_ollama': str(e)}


def verificar_ollama() -> bool:
    """Verifica se o Ollama está rodando e o modelo está disponível."""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            modelos = [m['name'] for m in r.json().get('models', [])]
            print(f"✅ Ollama online. Modelos disponíveis: {modelos}")
            if not any(OLLAMA_MODEL in m for m in modelos):
                print(f"⚠️  Modelo '{OLLAMA_MODEL}' não encontrado. Troque OLLAMA_MODEL na Célula 1.")
                return False
            return True
    except Exception:
        pass
    print("❌ Ollama offline. Rode: ollama serve")
    return False


OLLAMA_OK = verificar_ollama()

# Teste rápido se Ollama disponível
if OLLAMA_OK:
    print("\nTeste Ollama:")
    res = ollama_classificar_e_extrair("SWITCH 08 PORTAS 10/100/1000 MBPS INTELBRAS SG 800Q")
    print(res)

✅ Ollama online. Modelos disponíveis: ['qwen2.5:14b', 'deepseek-r1:7b', 'qwen2.5:7b', 'phi3:mini', 'llama3:70b', 'llama3:8b', 'llama3:latest', 'llama3.2:1b', 'mistral:7b-instruct-q4_0', 'llama2:latest', 'llama3:8b-instruct-q4_0']

Teste Ollama:
{'eh_principal_ollama': True, 'marca_ollama': 'Intelbras', 'modelo_ollama': 'SG 800Q', 'erro_ollama': None}
{'eh_principal_ollama': True, 'marca_ollama': 'Intelbras', 'modelo_ollama': 'SG 800Q', 'erro_ollama': None}


## 📥 Célula 6 — Carrega a Planilha de Entrada

In [19]:
# Lê o arquivo (suporta .xlsx, .xls e .csv)
if ARQUIVO_ENTRADA.endswith('.csv'):
    df_entrada = pd.read_csv(ARQUIVO_ENTRADA, dtype=str)
else:
    df_entrada = pd.read_excel(ARQUIVO_ENTRADA, dtype=str)

df_entrada = df_entrada.dropna(subset=[COLUNA_ID_PNCP])
df_entrada = df_entrada[df_entrada[COLUNA_ID_PNCP].str.strip() != '']
df_entrada = df_entrada.reset_index(drop=True)

print(f"✅ Planilha carregada: {len(df_entrada)} linhas")
print(f"Colunas: {list(df_entrada.columns)}")
display(df_entrada[[COLUNA_ID_PNCP]].head(5))

✅ Planilha carregada: 34 linhas
Colunas: ['Número ConLicitação', 'Código', 'Órgão', 'Endereço', 'Cidade', 'Estado', 'CEP', 'Edital', 'Site 1', 'Site 2', 'Processo', 'Valor Estimado', 'Itens', 'Situação', 'Documento', 'Abertura', 'Abertura 2', 'Prazo', 'Objeto', 'Observação', 'Anexos', 'Atualizada em', 'Comentários', 'ID PNCP', 'Tipo de licitação', 'Fonte', 'Status', 'Marca', 'Modelo', 'N Ata Tor']


,ID PNCP
0,14226731000164-1-000018/2025
1,18338285000130-1-000079/2025
2,58290502000184-1-000093/2025
3,46377800000127-1-004081/2025
4,96291141000180-1-004423/2025


## 🚀 Célula 7 — Pipeline Principal: Consulta API + Classificação

In [20]:
def _normalizar_texto(txt: str) -> str:
    txt = str(txt or "")
    txt = txt.lower()
    # normalizações simples para capturar variações comuns
    txt = txt.replace("–", "-").replace("—", "-")
    return txt.strip()

def contem_switch(descricao: str) -> bool:
    """Filtro leve para identificar itens que realmente falam de switch de rede."""
    t = _normalizar_texto(descricao)
    if not t:
        return False
    # pega switch / switches (plural)
    if not re.search(r"\bswitch(es)?\b", t):
        return False
    # evita falsos positivos comuns (ajuste conforme sua base)
    falsos_positivos = [
        r"\bswitch\s*(de)?\s*\bluz\b",
        r"\bswitch\s*(de)?\s*\bparede\b",
        r"\bswitch\s*hdmi\b",
        r"\bswitch\s*usb\b",
        r"\bswitch\s*kvm\b",
    ]
    for fp in falsos_positivos:
        if re.search(fp, t):
            return False
    return True

resultados = []
erros = []

ids_pncp = df_entrada[COLUNA_ID_PNCP].tolist()

for id_pncp in tqdm(ids_pncp, desc="Processando licitações"):

    # 1. Parse do ID
    params = parse_id_pncp(id_pncp)
    if not params:
        erros.append({'id_pncp': id_pncp, 'etapa': 'parse_id', 'erro': 'ID PNCP inválido'})
        continue

    cnpj, ano, seq = params['cnpj'], params['ano'], params['sequencial']

    # 2. Dados da contratação (com diagnóstico)
    url_contratacao = f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{seq}"
    contratacao, meta_contratacao = api_get_detalhado(url_contratacao)
    time.sleep(PAUSA_SEGUNDOS)

    if not contratacao or not isinstance(contratacao, dict):
        erros.append({
            'id_pncp': id_pncp,
            'etapa': 'contratacao',
            'url': url_contratacao,
            'status_code': meta_contratacao.get('status_code'),
            'erro': meta_contratacao.get('erro') or 'Contratação não encontrada/JSON inválido',
            'texto_curto': meta_contratacao.get('texto_curto'),
        })
        continue

    # 3. Itens da contratação (com diagnóstico)
    url_itens = f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{seq}/itens"
    itens_data, meta_itens = api_get_detalhado(url_itens)
    time.sleep(PAUSA_SEGUNDOS)

    itens = []
    if isinstance(itens_data, list):
        itens = itens_data
    elif isinstance(itens_data, dict) and 'data' in itens_data:
        itens = itens_data.get('data') or []

    if not itens:
        erros.append({
            'id_pncp': id_pncp,
            'etapa': 'itens',
            'url': url_itens,
            'status_code': meta_itens.get('status_code'),
            'erro': meta_itens.get('erro') or 'Nenhum item encontrado',
            'texto_curto': meta_itens.get('texto_curto'),
        })
        continue

    # 4. Filtra itens que contêm 'switch' na descrição (filtro refinado)
    itens_com_switch = [
        item for item in itens
        if contem_switch(item.get('descricao', ''))
    ]

    if not itens_com_switch:
        continue  # Nenhum item com switch nessa licitação

    # 5. Para cada item com switch — classifica e extrai
    for item in itens_com_switch:
        descricao    = str(item.get('descricao', ''))
        numero_item  = item.get('numeroItem')

        # --- Algoritmo 1: Regex ---
        eh_principal_regex = regex_eh_switch_principal(descricao)
        extracao_regex     = regex_extrair_marca_modelo(descricao)

        # --- Algoritmo 2: Ollama ---
        if OLLAMA_OK:
            extracao_ollama = ollama_classificar_e_extrair(descricao)
            time.sleep(0.2)
        else:
            extracao_ollama = {
                'eh_principal_ollama': None,
                'marca_ollama': None,
                'modelo_ollama': None,
                'erro_ollama': 'Ollama offline',
            }

        # --- Resultado do item (vencedor e preço homologado) ---
        url_result = f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{seq}/itens/{numero_item}/resultados"
        resultados_item_data, meta_res = api_get_detalhado(url_result)
        time.sleep(PAUSA_SEGUNDOS)

        resultados_item = resultados_item_data if isinstance(resultados_item_data, list) else []
        if (not resultados_item) and (meta_res.get('status_code') not in (None, 200)):
            erros.append({
                'id_pncp': id_pncp,
                'etapa': 'resultados_item',
                'numero_item': numero_item,
                'url': url_result,
                'status_code': meta_res.get('status_code'),
                'erro': meta_res.get('erro') or 'Falha ao buscar resultados do item',
                'texto_curto': meta_res.get('texto_curto'),
            })

        # Pega o primeiro resultado homologado (ordemClassificacao = 1)
        vencedor = next(
            (r for r in resultados_item if r.get('ordemClassificacaoSrp', 999) == 1),
            resultados_item[0] if resultados_item else {}
        )

        # --- Monta linha do resultado ---
        linha = {
            # Identificação
            'id_pncp':               id_pncp,
            'cnpj_orgao':            cnpj,
            'ano':                   ano,
            'sequencial':            seq,
            'numero_item':           numero_item,

            # Dados da contratação
            'orgao':                 contratacao.get('orgaoEntidade', {}).get('razaoSocial', ''),
            'municipio':             contratacao.get('unidadeOrgao', {}).get('municipioNome', ''),
            'uf':                    contratacao.get('unidadeOrgao', {}).get('ufSigla', ''),
            'modalidade':            contratacao.get('modalidadeNome', ''),
            'situacao_contratacao':  contratacao.get('situacaoCompraNome', ''),
            'data_abertura':         contratacao.get('dataAberturaProposta', ''),
            'data_publicacao':       contratacao.get('dataPublicacaoPncp', ''),
            'srp':                   contratacao.get('srp', ''),

            # Dados do item
            'descricao':             descricao,
            'material_servico':      item.get('materialOuServicoNome', ''),
            'quantidade':            item.get('quantidade', ''),
            'unidade_medida':        item.get('unidadeMedida', ''),
            'valor_unitario_est':    item.get('valorUnitarioEstimado', ''),
            'valor_total_est':       item.get('valorTotal', ''),
            'situacao_item':         item.get('situacaoCompraItemNome', ''),
            'tem_resultado':         item.get('temResultado', ''),
            'ncm':                   item.get('ncmNbsCodigo', ''),

            # Resultado (vencedor)
            'cnpj_vencedor':         vencedor.get('niFornecedor', ''),
            'nome_vencedor':         vencedor.get('nomeRazaoSocialFornecedor', ''),
            'porte_vencedor':        vencedor.get('porteFornecedorNome', ''),
            'valor_unit_homologado': vencedor.get('valorUnitarioHomologado', ''),
            'valor_total_homolog':   vencedor.get('valorTotalHomologado', ''),
            'qtd_homologada':        vencedor.get('quantidadeHomologada', ''),
            'data_resultado':        vencedor.get('dataResultado', ''),

            # Algoritmo 1 — Regex
            'switch_principal_regex': eh_principal_regex,
            'marca_regex':            extracao_regex.get('marca_regex'),
            'modelo_regex':           extracao_regex.get('modelo_regex'),

            # Algoritmo 2 — Ollama
            'switch_principal_ollama': extracao_ollama.get('eh_principal_ollama'),
            'marca_ollama':            extracao_ollama.get('marca_ollama'),
            'modelo_ollama':           extracao_ollama.get('modelo_ollama'),
            'erro_ollama':             extracao_ollama.get('erro_ollama'),
        }
        resultados.append(linha)

print(f"\n✅ Processamento concluído!")
print(f"   Itens com 'switch' encontrados: {len(resultados)}")
print(f"   Erros: {len(erros)}")

# Diagnóstico rápido dos erros
if erros:
    df_erros = pd.DataFrame(erros)
    display(df_erros.head(15))
    if 'status_code' in df_erros.columns:
        print("\nErros por status_code:")
        display(df_erros['status_code'].value_counts(dropna=False))

Processando licitações: 100%|██████████| 34/34 [00:20<00:00,  1.65it/s]


✅ Processamento concluído!
   Itens com 'switch' encontrados: 0
   Erros: 34


,id_pncp,etapa,url,status_code,erro,texto_curto
0,14226731000164-1-000018/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/1422673...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:45.078-03:00"",""..."
1,18338285000130-1-000079/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/1833828...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:45.684-03:00"",""..."
2,58290502000184-1-000093/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/5829050...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:46.302-03:00"",""..."
3,46377800000127-1-004081/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/4637780...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:46.908-03:00"",""..."
4,96291141000180-1-004423/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/9629114...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:47.496-03:00"",""..."
5,46634101000115-1-000401/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/4663410...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:48.105-03:00"",""..."
6,25944455000196-1-000100/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/2594445...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:48.727-03:00"",""..."
7,14006977000120-1-000135/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/1400697...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:49.340-03:00"",""..."
8,45699626000176-1-000393/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/4569962...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:49.985-03:00"",""..."
9,46137477000114-1-000097/2025,contratacao,https://pncp.gov.br/api/pncp/v1/orgaos/4613747...,301.0,Contratação não encontrada/JSON inválido,"{""timestamp"":""2026-03-19T20:31:50.604-03:00"",""..."



Erros por status_code:


status_code
301.0    33
NaN       1
Name: count, dtype: int64

## 📊 Célula 8 — Análise e Comparação dos Algoritmos

In [21]:
df = pd.DataFrame(resultados)

if df.empty:
    print("⚠️  Nenhum item com 'switch' encontrado.")
else:
    print(f"Total de itens com 'switch': {len(df)}")
    print()

    # ── Resumo por algoritmo ──────────────────────────────────────
    n_regex   = df['switch_principal_regex'].sum()
    n_ollama  = df['switch_principal_ollama'].sum() if OLLAMA_OK else 'N/A'

    print("=" * 50)
    print("CLASSIFICAÇÃO: switch como item PRINCIPAL")
    print("=" * 50)
    print(f"  Algoritmo 1 (Regex):  {n_regex} de {len(df)} itens ({100*n_regex/len(df):.1f}%)")
    if OLLAMA_OK:
        print(f"  Algoritmo 2 (Ollama): {n_ollama} de {len(df)} itens ({100*n_ollama/len(df):.1f}%)")
    print()

    # ── Concordância entre algoritmos ────────────────────────────
    if OLLAMA_OK:
        df_comp = df.dropna(subset=['switch_principal_ollama'])
        if len(df_comp) > 0:
            concordancia = (df_comp['switch_principal_regex'] == df_comp['switch_principal_ollama']).mean()
            print("=" * 50)
            print("CONCORDÂNCIA ENTRE ALGORITMOS")
            print("=" * 50)
            print(f"  Taxa de concordância: {concordancia*100:.1f}%")

            # Casos onde divergem
            divergencias = df_comp[
                df_comp['switch_principal_regex'] != df_comp['switch_principal_ollama']
            ][['descricao', 'switch_principal_regex', 'switch_principal_ollama']]

            print(f"  Divergências: {len(divergencias)} itens")
            if len(divergencias) > 0:
                print()
                print("Itens com divergência entre algoritmos:")
                display(divergencias.reset_index(drop=True))

    # ── Itens classificados como principal (por regex) ────────────
    print()
    print("=" * 50)
    print("ITENS CLASSIFICADOS COMO SWITCH PRINCIPAL (Regex)")
    print("=" * 50)
    df_principal = df[df['switch_principal_regex'] == True].copy()
    display(df_principal[['descricao', 'quantidade', 'valor_unitario_est',
                           'valor_unit_homologado', 'nome_vencedor',
                           'marca_regex', 'modelo_regex']].reset_index(drop=True))

    # ── Análise de preços (somente itens classificados como principal) ──
    if len(df_principal) > 0:
        df_precos = df_principal.copy()
        for col in ['valor_unitario_est', 'valor_unit_homologado']:
            df_precos[col] = pd.to_numeric(df_precos[col], errors='coerce')

        df_precos = df_precos.dropna(subset=['valor_unit_homologado'])

        if len(df_precos) > 0:
            print()
            print("=" * 50)
            print("ANÁLISE DE PREÇOS — SWITCH PRINCIPAL")
            print("=" * 50)
            stats = df_precos['valor_unit_homologado'].describe()
            print(f"  Menor preço:   R$ {stats['min']:,.2f}")
            print(f"  Maior preço:   R$ {stats['max']:,.2f}")
            print(f"  Média:         R$ {stats['mean']:,.2f}")
            print(f"  Mediana:       R$ {stats['50%']:,.2f}")
            print(f"  Qtd amostras:  {int(stats['count'])}")

            # Desconto médio estimado vs homologado
            df_desc = df_precos.dropna(subset=['valor_unitario_est'])
            df_desc = df_desc[df_desc['valor_unitario_est'] > 0]
            if len(df_desc) > 0:
                df_desc = df_desc.copy()
                df_desc['desconto_pct'] = (
                    (df_desc['valor_unitario_est'] - df_desc['valor_unit_homologado'])
                    / df_desc['valor_unitario_est'] * 100
                )
                print(f"  Desconto médio (est→homolog): {df_desc['desconto_pct'].mean():.1f}%")

⚠️  Nenhum item com 'switch' encontrado.


## 💾 Célula 9 — Exportação dos Resultados

In [22]:
if not df.empty:

    # ── Arquivo 1: Todos os dados extraídos ──────────────────────
    df.to_excel(ARQUIVO_SAIDA_DADOS, index=False)
    print(f"✅ Dados exportados: {ARQUIVO_SAIDA_DADOS}")

    # ── Arquivo 2: Relatório comparativo ─────────────────────────
    with pd.ExcelWriter(ARQUIVO_SAIDA_RELATORIO, engine='openpyxl') as writer:

        # Aba 1: Todos os itens com switch
        colunas_resumo = [
            'id_pncp', 'orgao', 'municipio', 'uf', 'descricao',
            'quantidade', 'unidade_medida',
            'valor_unitario_est', 'valor_unit_homologado', 'valor_total_homolog',
            'nome_vencedor', 'cnpj_vencedor', 'porte_vencedor',
            'data_resultado', 'situacao_item', 'srp',
            'switch_principal_regex', 'marca_regex', 'modelo_regex',
            'switch_principal_ollama', 'marca_ollama', 'modelo_ollama',
        ]
        colunas_existentes = [c for c in colunas_resumo if c in df.columns]
        df[colunas_existentes].to_excel(writer, sheet_name='Todos os Itens Switch', index=False)

        # Aba 2: Somente switch principal (regex)
        df_principal = df[df['switch_principal_regex'] == True]
        df_principal[colunas_existentes].to_excel(
            writer, sheet_name='Switch Principal (Regex)', index=False)

        # Aba 3: Somente switch principal (Ollama) — se disponível
        if OLLAMA_OK and 'switch_principal_ollama' in df.columns:
            df_principal_ollama = df[df['switch_principal_ollama'] == True]
            df_principal_ollama[colunas_existentes].to_excel(
                writer, sheet_name='Switch Principal (Ollama)', index=False)

        # Aba 4: Divergências
        if OLLAMA_OK and 'switch_principal_ollama' in df.columns:
            df_div = df[
                df['switch_principal_regex'] != df['switch_principal_ollama']
            ]
            if len(df_div) > 0:
                df_div[['id_pncp', 'descricao',
                         'switch_principal_regex', 'switch_principal_ollama',
                         'erro_ollama']].to_excel(
                    writer, sheet_name='Divergências', index=False)

        # Aba 5: Comparativo de preços
        df_precos = df[df['switch_principal_regex'] == True].copy()
        for col in ['valor_unitario_est', 'valor_unit_homologado']:
            df_precos[col] = pd.to_numeric(df_precos[col], errors='coerce')

        df_precos_clean = df_precos.dropna(subset=['valor_unit_homologado']).copy()
        df_precos_clean['desconto_%'] = (
            (df_precos_clean['valor_unitario_est'] - df_precos_clean['valor_unit_homologado'])
            / df_precos_clean['valor_unitario_est'].replace(0, float('nan')) * 100
        ).round(2)

        cols_precos = ['id_pncp', 'orgao', 'municipio', 'uf', 'descricao',
                       'marca_regex', 'modelo_regex', 'quantidade',
                       'valor_unitario_est', 'valor_unit_homologado', 'desconto_%',
                       'nome_vencedor', 'cnpj_vencedor', 'data_resultado']
        cols_ok = [c for c in cols_precos if c in df_precos_clean.columns]
        df_precos_clean[cols_ok].sort_values('valor_unit_homologado').to_excel(
            writer, sheet_name='Comparativo de Preços', index=False)

        # Aba 6: Erros
        if erros:
            pd.DataFrame(erros).to_excel(writer, sheet_name='Erros', index=False)

    print(f"✅ Relatório exportado: {ARQUIVO_SAIDA_RELATORIO}")
    print()
    print("Abas do relatório:")
    print("  📋 Todos os Itens Switch     — todos os itens onde 'switch' aparece na descrição")
    print("  ✅ Switch Principal (Regex)  — filtrado pelo Algoritmo 1")
    if OLLAMA_OK:
        print("  🤖 Switch Principal (Ollama) — filtrado pelo Algoritmo 2")
        print("  ⚠️  Divergências              — casos onde os algoritmos discordam")
    print("  💰 Comparativo de Preços     — preços ordenados do menor para o maior")
    if erros:
        print("  ❌ Erros                     — IDs que falharam")

else:
    print("⚠️  Nenhum resultado para exportar.")

⚠️  Nenhum resultado para exportar.
